# Eval Overview - Dose Response

As part of our training data, we have chemical perturbations that include different dosing of the same perturbation. While the dose is not passed to the model during training, we want to confirm whether our model has learned anything about dosing since we have it in our dataset. To do this evaluation, we built a category of benchmarks called "dose response" that test whether the model predicts stronger perturbation effects at higher drug concentrations. Since we currently only have 1 dataset with chemical perturbations, SciPlex, we only analyze that dataset. This analysis uses the per-sample expression delta produced by the [linear expression decoder](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_eval_decoders.ipynb). The goal of this evaluation is to see if, even without the dose information, our model has learned that higher drug concentrations produce stronger cellular effects. Since the input to our model across different dosages is identical (perturbation input does not include dosage), we would expect the predicted severity to be roughly flat across dose levels.

To evaluate dose response we calculate the monotonicity of predicted severity across dose levels and the Spearman rank correlation between dose and predicted severity. Note that we compare predicted severity against dose and do not currently evaluate against the real expression changes. We mainly do this because we are not training on dosage, so we would expect the performance to be quite bad. As you walk through the notebook, you'll see that we evaluate on multiple dimensions to ensure we have a thorough understanding of where our model is working well and where it's struggling.

In [1]:
import numpy as np
from scipy.stats import spearmanr

In [2]:
SEED = 1337
np.random.seed(SEED)

## Data Prep
We'll start by preparing our data. For this evaluation, we need predicted expression deltas, dose values, and perturbation keys for each sample. A "perturbation" is a unique combination of a sequence, target, modality, and mode applied to a cell type. A perturbation can span datasets, though since we're focusing on chemical perturbations only, currently all perturbations are isolated to a single dataset. To show our analysis, we'll need multiple cells that have the same perturbation with different dosages. We'll mock up 3 chemical perturbations across 10 samples to show distinct dose-response behaviors.

For predicted expression, we use the data created by the [linear expression decoder](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_eval_decoders.ipynb). As a reminder, the decoder takes the mean ($\mu$) BioJEPA-AC output based on the perturbations and cell expression pattern, and then uses a linear layer to project down to a $[\text{n\_genes},\text{1}]$ matrix with a single value per gene representing the expression.

We'll stage the data to show a few different response behaviors: monotonic (severity increases consistently with dose), partial monotonic, and non-monotonic. Since our [gene expression prediction explainer notebook](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_eval_expr_prediction.ipynb) and others have walked through how sample-level deltas are computed, we'll stage the deltas directly.

In [3]:
num_genes = 8
num_samples = 10
unique_perts = 3

**Perturbations**

We'll first start with our perturbations. Since we're focused on dose response, our perturbations will all be chemical. To define a unique perturbation, it's not just about what we target, but the context of it. Because of this, we represent a unique perturbation as $\text{(seq id, targ id, modality id, mode id, cell type)}$. Notice that the dose is not in the perturbation so perturbations across doses are considered the same perturbation. IDs are used since our model keeps the perturbation information in separate caches from our sample expression counts to avoid heavy duplication of information.

We'll also create a mapping of the samples to the perturbations where the first 4 samples belong to the first perturbation, and the last 6 are split evenly across the two remaining perturbations. We want multiple samples per perturbation to show the effects of different doses.

In [4]:
pert_keys = [
    (0,0,2,4,0),  # Drug A: chemical inhibitor, cell type 0
    (1,1,2,4,0),  # Drug B: chemical inhibitor, cell type 0
    (2,2,2,4,0),  # Drug C: chemical inhibitor, cell type 0
]

sample_to_pert = [0,0,0,0,1,1,1,2,2,2]
sample_to_pert

[0, 0, 0, 0, 1, 1, 1, 2, 2, 2]

**Doses**

Each sample has an associated dose value representing the drug concentration applied to that cell. The doses are stored in our training shards so we can pull them out per sample. For our eval, we only focus on valid dose data so any sentinel values (-1.0, meaning no dose info available) are filtered out. Also, we currently focus only on single drug perturbations. For our dosage, we'll use a positional list where each position corresponds to a sample. If we mirror this with our perturbations, you'll see that we have staged doses of $[0.1, 1, 10]$ for each perturbation, though we have two samples at dose 1.0 for our first perturbation.

In [5]:
sample_doses = np.array([0.1,1.0,1.0,10.0,0.1,1.0,10.0,0.1,1.0,10.0])
sample_doses.shape, sample_doses

((10,), array([ 0.1,  1. ,  1. , 10. ,  0.1,  1. , 10. ,  0.1,  1. , 10. ]))

**Predicted Expression Deltas**

To do our evaluation, we also need to know for each sample, how much the model predicts the expression will change. This requires only the control expression and the perturbation, which is why we do not expect dosage to have much impact on our real data. Expression change is calculated as $\hat{\delta}_g = \hat{x}^{\text{case}}_g - \hat{x}^{\text{ctrl}}_g$, the predicted perturbed expression minus the predicted control expression. We use the predicted control expression to isolate BioJEPA-AC's learned perturbation effect from any baseline reconstruction error. Since we have a few different notebooks showing how sample-level deltas are computed, we'll stage the deltas directly.

We've staged our expression deltas to show three behaviors: Drug A increases with dose (monotonic), Drug B peaks at the middle dose (partial), and Drug C stays roughly flat (non-monotonic).

In [6]:
pred_deltas = np.array([
    [0.1,-0.2,0.1,0.0,0.1,0.0,-0.1,0.0],        # Drug A, dose 0.1
    [0.4,-0.5,0.3,0.2,0.3,-0.1,-0.4,0.1],        # Drug A, dose 1.0
    [0.5,-0.4,0.4,0.1,0.2,-0.2,-0.3,0.2],        # Drug A, dose 1.0
    [1.0,-0.8,0.7,0.5,0.6,-0.3,-0.9,0.4],        # Drug A, dose 10.0
    [0.05,-0.1,0.05,0.1,0.0,0.0,-0.05,0.0],      # Drug B, dose 0.1
    [0.8,-0.7,0.5,0.4,0.3,-0.2,-0.6,0.3],        # Drug B, dose 1.0
    [0.5,-0.3,0.3,0.2,0.2,-0.1,-0.4,0.2],        # Drug B, dose 10.0
    [0.3,-0.3,0.2,0.1,0.2,-0.1,-0.2,0.1],        # Drug C, dose 0.1
    [0.3,-0.2,0.25,0.05,0.15,-0.15,-0.25,0.15],  # Drug C, dose 1.0
    [0.25,-0.25,0.15,0.15,0.1,-0.1,-0.15,0.15],  # Drug C, dose 10.0
])
pred_deltas.shape, pred_deltas

((10, 8),
 array([[ 0.1 , -0.2 ,  0.1 ,  0.  ,  0.1 ,  0.  , -0.1 ,  0.  ],
        [ 0.4 , -0.5 ,  0.3 ,  0.2 ,  0.3 , -0.1 , -0.4 ,  0.1 ],
        [ 0.5 , -0.4 ,  0.4 ,  0.1 ,  0.2 , -0.2 , -0.3 ,  0.2 ],
        [ 1.  , -0.8 ,  0.7 ,  0.5 ,  0.6 , -0.3 , -0.9 ,  0.4 ],
        [ 0.05, -0.1 ,  0.05,  0.1 ,  0.  ,  0.  , -0.05,  0.  ],
        [ 0.8 , -0.7 ,  0.5 ,  0.4 ,  0.3 , -0.2 , -0.6 ,  0.3 ],
        [ 0.5 , -0.3 ,  0.3 ,  0.2 ,  0.2 , -0.1 , -0.4 ,  0.2 ],
        [ 0.3 , -0.3 ,  0.2 ,  0.1 ,  0.2 , -0.1 , -0.2 ,  0.1 ],
        [ 0.3 , -0.2 ,  0.25,  0.05,  0.15, -0.15, -0.25,  0.15],
        [ 0.25, -0.25,  0.15,  0.15,  0.1 , -0.1 , -0.15,  0.15]]))

## Dose-Response Evaluation

Now we'll evaluate whether our predicted expression changes scale with dose. To do this, we first convert our expression delta into scalar severity scores, take the per-perturbation and dose averages, and then evaluate the correlation with dosage.

### Severity Calculation

We'll start by calculating the severity per sample. We calculate severity as the L2 norm of the expression delta vector. This collapses the per-gene predicted changes into a single number representing the overall magnitude of the perturbation effect. We calculate it as:
$$
\text{severity}_s = \|\hat{\delta}_s\| = \sqrt{\sum_{g=1}^{G} \hat{\delta}_{s,g}^2}
$$

This uses only predicted deltas, not real deltas, because we're testing whether the model's own predictions scale with dose, not whether reality does. Because of how we staged our data, you'll see Drug A's severity climbing from the low dose to the high dose, Drug B peaking at the middle dose, and Drug C staying roughly flat.

In [7]:
severity = np.linalg.norm(pred_deltas, axis=1)
severity.shape, severity

((10,),
 array([0.28284271, 0.9       , 0.88881944, 1.94935887, 0.16583124,
        1.45602198, 0.84852814, 0.57445626, 0.57008771, 0.48476799]))

**Per-Perturbation & Dose Average**

Now that we have a single severity per sample, we need to group samples by drug and dose level. Using our `sample_to_pert` mapping, we can identify which drug each sample received, then pair it with the dose and severity. Within each drug, we further group by dose level and average the severity across replicates. We calculate the per-dose-level mean severity as:
$$
\bar{s}_{p,d} = \frac{1}{|R_{p,d}|}\sum_{s \in R_{p,d}} \text{severity}_s
$$

where $R_{p,d}$ is the set of samples for drug $p$ at dose level $d$. For the first perturbation, we have two samples at dose 1.0 that will be averaged into a single severity estimate, reducing noise from cell-to-cell variability.

In [8]:
pert_dose_sev = {}

In [9]:
for i in range(num_samples):
    print(f'---- sample {i} ----')
    pert_idx = sample_to_pert[i]
    dosage = sample_doses[i]
    sev = severity[i]
    print(f'pert_id {pert_idx} | dosage {dosage} | sev {sev}')

    if pert_idx not in pert_dose_sev:
        pert_dose_sev[pert_idx] = {}
    if dosage not in pert_dose_sev[pert_idx]:
        pert_dose_sev[pert_idx][dosage] = []
    pert_dose_sev[pert_idx][dosage].append(sev)

pert_dose_sev

---- sample 0 ----
pert_id 0 | dosage 0.1 | sev 0.28284271247461906
---- sample 1 ----
pert_id 0 | dosage 1.0 | sev 0.9
---- sample 2 ----
pert_id 0 | dosage 1.0 | sev 0.8888194417315589
---- sample 3 ----
pert_id 0 | dosage 10.0 | sev 1.9493588689617927
---- sample 4 ----
pert_id 1 | dosage 0.1 | sev 0.16583123951777
---- sample 5 ----
pert_id 1 | dosage 1.0 | sev 1.4560219778561037
---- sample 6 ----
pert_id 1 | dosage 10.0 | sev 0.848528137423857
---- sample 7 ----
pert_id 2 | dosage 0.1 | sev 0.5744562646538028
---- sample 8 ----
pert_id 2 | dosage 1.0 | sev 0.570087712549569
---- sample 9 ----
pert_id 2 | dosage 10.0 | sev 0.4847679857416329


{0: {np.float64(0.1): [np.float64(0.28284271247461906)],
  np.float64(1.0): [np.float64(0.9), np.float64(0.8888194417315589)],
  np.float64(10.0): [np.float64(1.9493588689617927)]},
 1: {np.float64(0.1): [np.float64(0.16583123951777)],
  np.float64(1.0): [np.float64(1.4560219778561037)],
  np.float64(10.0): [np.float64(0.848528137423857)]},
 2: {np.float64(0.1): [np.float64(0.5744562646538028)],
  np.float64(1.0): [np.float64(0.570087712549569)],
  np.float64(10.0): [np.float64(0.4847679857416329)]}}

**Mean per perturbation & dose**

Now we'll calculate the mean. If you look, you can see that for the first perturbation, we've staged an increase as dosage increased, the second jumps around, and the third decreases.

In [10]:
avg_sev = {p: {d: np.mean(sevs) for d, sevs in doses.items()} for p, doses in pert_dose_sev.items()}
avg_sev

{0: {np.float64(0.1): np.float64(0.28284271247461906),
  np.float64(1.0): np.float64(0.8944097208657795),
  np.float64(10.0): np.float64(1.9493588689617927)},
 1: {np.float64(0.1): np.float64(0.16583123951777),
  np.float64(1.0): np.float64(1.4560219778561037),
  np.float64(10.0): np.float64(0.848528137423857)},
 2: {np.float64(0.1): np.float64(0.5744562646538028),
  np.float64(1.0): np.float64(0.570087712549569),
  np.float64(10.0): np.float64(0.4847679857416329)}}

### Monotonicity Score

Now we're ready to evaluate what fraction of consecutive dose increases actually produce an increase in predicted severity. This monotonicity evaluation helps us understand if predicted severity increases with dosage. We calculate it as:
$$
\text{Monotonicity} = \frac{1}{T}\sum_{t=1}^{T}\mathbf{1}[\bar{s}_{p,d_{t+1}} > \bar{s}_{p,d_t}]
$$
where $T$ is the total number of consecutive dose-level pairs across all drugs. Because of how we staged the data, our first perturbation will show perfect monotonicity, our second will only show it for the first dosage change but not the second, and our third, since it decreases, will show 0. Note that 0 is useful also since it highlights an inverse correlation between severity and dosage.

In [11]:
total_mon_count = 0.0
total_pairs = 0.0

In [12]:
for p in avg_sev:
    print(f'---- pert {p} ----')
    sorted_doses = sorted(avg_sev[p].keys())
    sev = [avg_sev[p][d] for d in sorted_doses]
    print(sev)
    mono_counts = sum(1 for i in range(len(sev) - 1) if sev[i + 1] > sev[i])
    pairs = len(sev) - 1
    total_mon_count += mono_counts
    total_pairs += pairs

    monotonicity = mono_counts / pairs
    print(f'monotonicity count {mono_counts} | pairs {pairs} | monotonicity {monotonicity}')

---- pert 0 ----
[np.float64(0.28284271247461906), np.float64(0.8944097208657795), np.float64(1.9493588689617927)]
monotonicity count 2 | pairs 2 | monotonicity 1.0
---- pert 1 ----
[np.float64(0.16583123951777), np.float64(1.4560219778561037), np.float64(0.848528137423857)]
monotonicity count 1 | pairs 2 | monotonicity 0.5
---- pert 2 ----
[np.float64(0.5744562646538028), np.float64(0.570087712549569), np.float64(0.4847679857416329)]
monotonicity count 0 | pairs 2 | monotonicity 0.0


**Overall Monotonicity**

Now we're ready to get the overall monotonicity score. If you look at our loop, this is actually a pair-weighted mean, so if one perturbation has more dose levels than the others, it will contribute more.

In [13]:
monotonicity_score = float(total_mon_count / total_pairs)
monotonicity_score

0.5

### Spearman Rank Correlation

Monotonicity gives us a count of how often severity increases when dose increases, but we also want to know if, across our dataset, predicted severity is correlated with dose. We use Spearman correlation to measure whether the rank ordering of doses matches the rank ordering of mean severities. We calculate it as:
$$
\rho = 1 - \frac{6\sum_{i=1}^{n}(R(d_i) - R(s_i))^2}{n(n^2 - 1)}
$$
where $R(d_i)$ and $R(s_i)$ are the ranks of the dose and severity values. We compute this across all dose-level means from all drugs pooled together, so each perturbation contributes its own set of (dose, severity) points. Because of how we staged our data, we'll see a moderate positive correlation since our first perturbation is strongly monotonic, which pulls the correlation up.

In [14]:
all_doses_flat = [d for p in avg_sev for d in sorted(avg_sev[p].keys())]
all_doses_flat

[np.float64(0.1),
 np.float64(1.0),
 np.float64(10.0),
 np.float64(0.1),
 np.float64(1.0),
 np.float64(10.0),
 np.float64(0.1),
 np.float64(1.0),
 np.float64(10.0)]

In [15]:
all_severities_flat = [avg_sev[p][d] for p in avg_sev for d in sorted(avg_sev[p].keys())]
all_severities_flat

[np.float64(0.28284271247461906),
 np.float64(0.8944097208657795),
 np.float64(1.9493588689617927),
 np.float64(0.16583123951777),
 np.float64(1.4560219778561037),
 np.float64(0.848528137423857),
 np.float64(0.5744562646538028),
 np.float64(0.570087712549569),
 np.float64(0.4847679857416329)]

In [16]:
r, _ = spearmanr(all_doses_flat, all_severities_flat)
dose_severity_spearman = float(r)
dose_severity_spearman

0.5270462766947299

## Dose Response Final Wrapup

We've now walked through the dose response evaluation that tests whether a model's predicted severity scales with perturbation dose. As a reminder, since BioJEPA-AC does not take in dose, we expect these values to be close to flat across dose levels. If they are not, this likely means there is some other correlation in the dataset that the model has learned, which would be a concern worth investigating for other evals.